In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

Demonstrando como os dados podem ser carregados e utilizados para tarefas de ML:

In [ ]:
# 1. carrega metadados e embeddings
df = pd.read_csv('piguard_sample_5k.csv')

df.head()

,text,label,source,id
0,Do crafts make your home look nicer?,0,open-instruct,0
1,Rewrite the given text to make it more formal....,1,TaskTracker,1
2,"Given the sentence ""People eating at a long ta...",0,safe-guard-prompt-injection,2
3,Task: Classify each word as one of the followi...,0,safe-guard-prompt-injection,3
4,"Abala, ţi abala, I want to learn more about th...",0,over-defense,4


In [ ]:
# escolhe o modelo de embedding que quer usar
bert = np.load('embeddings/bert.npy')
# roberta = np.load('embeddings/roberta.npy')
# ...

print(f"Dataset: {df.shape}")
print(f"Embeddings: {bert.shape}")
print(f"Exemplo id=0: label={df.iloc[0]['label']}, "
      f"embedding shape={bert[0].shape}")

Dataset: (5000, 4)
Embeddings: (5000, 768)
Exemplo id=0: label=0, embedding shape=(768,)


In [4]:
# 2. prepara X e y
X = bert
y = df['label'].astype(int).values

# 3. split treino/teste
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, df.index.values,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
df_train = df.iloc[idx_train].reset_index(drop=True)
df_test  = df.iloc[idx_test].reset_index(drop=True)

print(f"\nTreino: {X_train.shape[0]} exemplos "
      f"({y_train.sum()} comprometidos, {(y_train==0).sum()} benignos)")
print(f"Teste:  {X_test.shape[0]} exemplos "
      f"({y_test.sum()} comprometidos, {(y_test==0).sum()} benignos)")



Treino: 4000 exemplos (817 comprometidos, 3183 benignos)
Teste:  1000 exemplos (204 comprometidos, 796 benignos)


In [6]:
# treinando regressão logística simples para demonstração
clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print("\nResultados (BERT + Logistic Regression):")
print(classification_report(y_test, y_pred,
                             target_names=['benigno', 'comprometido']))

errors = df_test[y_pred != y_test].copy()
errors['predicted'] = y_pred[y_pred != y_test]


Resultados (BERT + Logistic Regression):
              precision    recall  f1-score   support

     benigno       0.94      0.96      0.95       796
comprometido       0.83      0.78      0.80       204

    accuracy                           0.92      1000
   macro avg       0.89      0.87      0.88      1000
weighted avg       0.92      0.92      0.92      1000

